# VéloCité – De l'API au tableau de bord
**Projet individuel Data Analyst** · Pipeline automatisé de collecte des disponibilités Vélib'

**Sommaire**
1. Cadrage du besoin
2. Installation
3. Exploration de l'API
4. Nettoyage et transformations
5. Stockage dans BigQuery
6. Script de collecte automatique (`src/collecte.py`)
7. Test du script
8. Automatisation
9. Requêtes SQL
10. Publication sur GitHub
11. Contrôles qualité
12. Limites et pistes d'amélioration

## 1. Cadrage du besoin

**Entreprise :** VéloCité, opérateur fictif de vélos en libre-service en Île-de-France.
**Problème :** des stations **vides** (impossible de prendre un vélo) ou **pleines** (impossible de le rendre).
**Question métier :** *quand et où les stations sont-elles vides ou pleines, pour mieux organiser le rééquilibrage et la maintenance ?*

**API choisie :** open data **Vélib' Métropole** (format standard GBFS) : publique, gratuite, sans clé, mise à jour en temps réel pour environ 1 500 stations. Deux flux complémentaires reliés par `station_id` :
- `station_information` : nom, position GPS, capacité, code station
- `station_status` : vélos disponibles (mécaniques / électriques), places libres, état

**Fréquence :** une collecte **toutes les heures**, pour construire un historique (l'API ne donne que l'instant présent).

## 2. Installation

Bibliothèques nécessaires (listées dans `requirements.txt`) :
```bash
pip install -r requirements.txt
```
Connexion à BigQuery : au premier envoi, `pandas-gbq` ouvre le navigateur pour se connecter avec son compte Google (aucune clé stockée dans le projet).

## 3. Exploration de l'API
### 3.1 Bibliothèques et adresses de l'API

In [1]:
import requests
import pandas as pd

URL_INFO = "https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_information.json"
URL_STATUS = "https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_status.json"

### 3.2 Flux `station_information` : contrôle du code HTTP
Le code **200** signifie que l'API a bien répondu ; tout autre code est traité comme une erreur.

In [2]:
reponse = requests.get(URL_INFO, timeout=30)
print("Code HTTP :", reponse.status_code)

if reponse.status_code == 200:
    stations = reponse.json()["data"]["stations"]
    print(len(stations), "stations récupérées")
else:
    print("Erreur :", reponse.status_code)

Code HTTP : 200
1519 stations récupérées


In [43]:
df_info = pd.DataFrame(stations)
df_info.head()

,station_id,stationCode,name,lat,lon,capacity,station_opening_hours,rental_methods
0,213688169,16107,Benjamin Godard - Victor Hugo,48.865983,2.275725,35,None,NaN
1,19179944124,40001,Hôpital Mondor,48.798922,2.453745,28,None,NaN
2,18795078746,32304,Charcot - Benfleet,48.878370,2.440524,28,None,NaN
3,36255,9020,Toudouze - Clauzel,48.879296,2.337360,21,None,[CREDITCARD]
4,251039991,14111,Cassini - Denfert-Rochereau,48.837526,2.336035,25,None,[CREDITCARD]


In [44]:
df_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1519 entries, 0 to 1518
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   station_id             1519 non-null   int64  
 1   stationCode            1519 non-null   object 
 2   name                   1519 non-null   object 
 3   lat                    1519 non-null   float64
 4   lon                    1519 non-null   float64
 5   capacity               1519 non-null   int64  
 6   station_opening_hours  0 non-null      object 
 7   rental_methods         894 non-null    object 
dtypes: float64(2), int64(2), object(4)
memory usage: 95.1+ KB


**Observations :** 1 519 stations · `station_opening_hours` est entièrement vide · `rental_methods` manque pour 625 stations (pas de paiement par carte) · `stationCode` est un texte dont les premiers chiffres donnent l'arrondissement.

### 3.3 Doublons
Contrôle sur la clé unique `station_id` (et non sur le nom : deux stations différentes s'appellent « Place Nelson Mandela »).

In [45]:
df_info["station_id"].duplicated().sum()

np.int64(0)

### 3.4 Flux `station_status`

In [46]:
reponse_status = requests.get(URL_STATUS, timeout=30)
reponse_status.status_code

200

In [47]:
df_status = pd.DataFrame(reponse_status.json()["data"]["stations"])
df_status.head()

,station_id,num_bikes_available,numBikesAvailable,num_bikes_available_types,num_docks_available,numDocksAvailable,is_installed,is_returning,is_renting,last_reported,stationCode,station_opening_hours
0,213688169,5,5,"[{'mechanical': 5}, {'ebike': 0}]",30,30,1,1,1,1792259588,16107,None
1,19179944124,13,13,"[{'mechanical': 8}, {'ebike': 5}]",15,15,1,1,1,1792259700,40001,None
2,18795078746,6,6,"[{'mechanical': 1}, {'ebike': 5}]",23,23,1,1,1,1792259378,32304,None
3,36255,0,0,"[{'mechanical': 0}, {'ebike': 0}]",21,21,1,1,1,1792259634,9020,None
4,251039991,4,4,"[{'mechanical': 0}, {'ebike': 4}]",20,20,1,1,1,1792259725,14111,None


In [48]:
df_status.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1519 entries, 0 to 1518
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   station_id                 1519 non-null   int64 
 1   num_bikes_available        1519 non-null   int64 
 2   numBikesAvailable          1519 non-null   int64 
 3   num_bikes_available_types  1519 non-null   object
 4   num_docks_available        1519 non-null   int64 
 5   numDocksAvailable          1519 non-null   int64 
 6   is_installed               1519 non-null   int64 
 7   is_returning               1519 non-null   int64 
 8   is_renting                 1519 non-null   int64 
 9   last_reported              1519 non-null   int64 
 10  stationCode                1519 non-null   object
 11  station_opening_hours      0 non-null      object
dtypes: int64(9), object(3)
memory usage: 142.5+ KB


**Observations :** même nombre de stations que `df_info` · colonnes en double (`numBikesAvailable`, `numDocksAvailable`) · `num_bikes_available_types` est imbriqué (liste de dictionnaires) · `last_reported` est un timestamp Unix.

## 4. Nettoyage et transformations

### 4.1 Nettoyage de `df_status`

In [49]:
df_status = df_status.drop(columns=["numBikesAvailable", "numDocksAvailable", "station_opening_hours", "stationCode"])

df_status["nb_mecaniques"] = df_status["num_bikes_available_types"].str[0].str["mechanical"]
df_status["nb_electriques"] = df_status["num_bikes_available_types"].str[1].str["ebike"]
df_status = df_status.drop(columns=["num_bikes_available_types"])

df_status["last_reported"] = pd.to_datetime(df_status["last_reported"], unit="s", utc=True).dt.tz_convert("Europe/Paris")
df_status.head()

,station_id,num_bikes_available,num_docks_available,is_installed,is_returning,is_renting,last_reported,nb_mecaniques,nb_electriques
0,213688169,5,30,1,1,1,2026-10-17 19:53:08+02:00,5,0
1,19179944124,13,15,1,1,1,2026-10-17 19:55:00+02:00,8,5
2,18795078746,6,23,1,1,1,2026-10-17 19:49:38+02:00,1,5
3,36255,0,21,1,1,1,2026-10-17 19:53:54+02:00,0,0
4,251039991,4,20,1,1,1,2026-10-17 19:55:25+02:00,0,4


### 4.2 Nettoyage de `df_info`

In [50]:
df_info["accepte_cb"] = df_info["rental_methods"].notna()
df_info = df_info.drop(columns=["station_opening_hours", "rental_methods"])
df_info.head()

,station_id,stationCode,name,lat,lon,capacity,accepte_cb
0,213688169,16107,Benjamin Godard - Victor Hugo,48.865983,2.275725,35,False
1,19179944124,40001,Hôpital Mondor,48.798922,2.453745,28,False
2,18795078746,32304,Charcot - Benfleet,48.878370,2.440524,28,False
3,36255,9020,Toudouze - Clauzel,48.879296,2.337360,21,True
4,251039991,14111,Cassini - Denfert-Rochereau,48.837526,2.336035,25,True


### 4.3 Jointure des deux flux
Équivalent d'un `LEFT JOIN` SQL sur `station_id`.

In [51]:
df = df_status.merge(df_info, on="station_id", how="left")
df.shape

(1519, 15)

### 4.4 Indicateurs

In [52]:
df = df[(df["capacity"] > 0) & (df["is_installed"] == 1)]
df["date_collecte"] = pd.Timestamp.now(tz="Europe/Paris")
df["taux_remplissage"] = df["num_bikes_available"] / df["capacity"]
df["station_vide"] = df["num_bikes_available"] == 0
df["station_pleine"] = df["num_docks_available"] == 0
df.shape

(1516, 19)

In [53]:
print("Stations vides :", df["station_vide"].sum())
print("Stations pleines :", df["station_pleine"].sum())

Stations vides : 94
Stations pleines : 38


## 5. Stockage dans BigQuery

- **Projet Google Cloud :** `velocite-velib` (bac à sable gratuit)
- **Dataset :** `velib` (région `europe-west9`, Paris)
- **Modèle de données :**

| Table / vue | Contenu | Mise à jour |
|---|---|---|
| `stations` | Infos fixes : nom, GPS, capacité, code, paiement CB | Remplacée à chaque collecte |
| `disponibilites` | Une photo de chaque station par heure | Ajout (append) avec anti-doublons |
| `journal_executions` | Une ligne par lancement : date, statut, nombre de lignes | Ajout |
| `vue_dashboard` | Jointure + champs prêts pour Data Studio | Vue (toujours à jour) |

L'envoi est fait par le script de la partie 6 (`pandas_gbq.to_gbq`).

## 6. Script de collecte automatique (`src/collecte.py`)

In [54]:
import os
os.makedirs("src", exist_ok=True)

### Partie 1 – Configuration
Bibliothèques, identifiants du projet BigQuery et adresses de l'API.

In [55]:
%%writefile src/collecte.py
"""Collecte des disponibilités Vélib' et envoi vers BigQuery."""
import logging
from pathlib import Path

import pandas as pd
import pandas_gbq
import requests

PROJET = "velocite-velib"
DATASET = "velib"
URL_INFO = "https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_information.json"
URL_STATUS = "https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_status.json"

Overwriting src/collecte.py


### Partie 2 – Traces d'exécution (logs)
Chaque lancement est enregistré dans `logs/collecte.log` (date, niveau, message).

In [56]:
%%writefile -a src/collecte.py


DOSSIER_LOGS = Path(__file__).resolve().parent.parent / "logs"
DOSSIER_LOGS.mkdir(exist_ok=True)
logging.basicConfig(
    filename=DOSSIER_LOGS / "collecte.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    encoding="utf-8",
)

Appending to src/collecte.py


### Partie 3 – Appel de l'API
Le code HTTP est écrit dans les logs ; s'il n'est pas **200**, la collecte s'arrête et l'erreur est tracée.

In [57]:
%%writefile -a src/collecte.py


def appeler_api(url):
    """Appelle l'API, vérifie le code HTTP et renvoie les stations en DataFrame."""
    reponse = requests.get(url, timeout=30)
    nom_flux = url.split("/")[-1]
    logging.info(f"Appel API {nom_flux} : code HTTP {reponse.status_code}")
    if reponse.status_code != 200:
        raise ValueError(f"Réponse inattendue de l'API ({nom_flux}) : code {reponse.status_code}")
    return pd.DataFrame(reponse.json()["data"]["stations"])

Appending to src/collecte.py


### Partie 4 – Table `stations`
Informations fixes des stations ; `rental_methods` devient le booléen `accepte_cb`.

In [58]:
%%writefile -a src/collecte.py


def preparer_stations(df_info):
    """Table stations : infos fixes de chaque station."""
    df_info["accepte_cb"] = df_info["rental_methods"].notna()
    colonnes = ["station_id", "stationCode", "name", "lat", "lon", "capacity", "accepte_cb"]
    return df_info[colonnes].drop_duplicates(subset="station_id")

Appending to src/collecte.py


### Partie 5 – Table `disponibilites`
Aplatissement des types de vélos, conversion des dates, clé unique **`station_id` + heure de collecte** (une photo par station et par heure), exclusion des stations à capacité 0 ou non installées, puis calcul des indicateurs.

In [59]:
%%writefile -a src/collecte.py


def preparer_disponibilites(df_status, df_stations):
    """Table disponibilites : une photo de chaque station par heure."""
    df = df_status.copy()
    df["nb_mecaniques"] = df["num_bikes_available_types"].str[0].str["mechanical"]
    df["nb_electriques"] = df["num_bikes_available_types"].str[1].str["ebike"]

    df["last_reported"] = pd.to_datetime(df["last_reported"], unit="s", utc=True)
    df["date_collecte"] = pd.Timestamp.now(tz="UTC")
    # Clé unique : station + heure de collecte (ex. 213688169_2026101309)
    df["cle"] = df["station_id"].astype(str) + "_" + df["date_collecte"].dt.strftime("%Y%m%d%H")

    df = df.merge(df_stations[["station_id", "capacity"]], on="station_id", how="left")
    df = df[(df["capacity"] > 0) & (df["is_installed"] == 1)]
    df["taux_remplissage"] = df["num_bikes_available"] / df["capacity"]
    df["station_vide"] = df["num_bikes_available"] == 0
    df["station_pleine"] = df["num_docks_available"] == 0

    colonnes = ["cle", "station_id", "num_bikes_available", "num_docks_available",
                "nb_mecaniques", "nb_electriques", "is_renting", "is_returning",
                "last_reported", "date_collecte", "taux_remplissage",
                "station_vide", "station_pleine"]
    return df[colonnes].drop_duplicates(subset="cle")

Appending to src/collecte.py


### Partie 6 – Anti-doublons
Avant l'envoi, on récupère les clés déjà présentes dans BigQuery et on ne garde que les nouvelles lignes.

In [60]:
%%writefile -a src/collecte.py


def retirer_deja_presentes(df):
    """Anti-doublons : on retire les lignes dont la clé est déjà dans BigQuery."""
    requete = f"SELECT cle FROM `{PROJET}.{DATASET}.disponibilites`"
    try:
        deja = pandas_gbq.read_gbq(requete, project_id=PROJET)
    except Exception:
        logging.info("Table disponibilites absente : première collecte")
        return df
    return df[~df["cle"].isin(deja["cle"])]

Appending to src/collecte.py


### Partie 7 – Journal des exécutions
Chaque lancement ajoute une ligne dans la table BigQuery `journal_executions` : preuve que l'automatisation tourne.

In [61]:
%%writefile -a src/collecte.py


def ecrire_journal(statut, nb_lignes, message):
    """Trace de chaque exécution dans la table journal_executions."""
    journal = pd.DataFrame([{
        "date_execution": pd.Timestamp.now(tz="UTC"),
        "statut": statut,
        "nb_lignes": nb_lignes,
        "message": message,
    }])
    pandas_gbq.to_gbq(journal, f"{DATASET}.journal_executions", project_id=PROJET, if_exists="append")

Appending to src/collecte.py


### Partie 8 – Programme principal
Appel → préparation → anti-doublons → envoi dans BigQuery → journal. En cas d'erreur : trace dans les logs et dans le journal.

In [67]:
%%writefile -a src/collecte.py


def main():
    logging.info("Début de la collecte")
    try:
        df_stations = preparer_stations(appeler_api(URL_INFO))
        df_dispo = preparer_disponibilites(appeler_api(URL_STATUS), df_stations)
        nb_recues = len(df_dispo)
        df_dispo = retirer_deja_presentes(df_dispo)

        pandas_gbq.to_gbq(df_stations, f"{DATASET}.stations", project_id=PROJET, if_exists="replace")
        if len(df_dispo) > 0:
            pandas_gbq.to_gbq(df_dispo, f"{DATASET}.disponibilites", project_id=PROJET, if_exists="append")

        message = f"{nb_recues} lignes reçues, {len(df_dispo)} nouvelles envoyées"
        logging.info(message)
        ecrire_journal("OK", len(df_dispo), message)
    except Exception as erreur:
        logging.error(f"Échec de la collecte : {erreur}")
        try:
            ecrire_journal("ERREUR", 0, str(erreur))
        except Exception:
            pass
        raise


if __name__ == "__main__":
    main()

Appending to src/collecte.py


## 7. Test du script
Lancé deux fois dans la même heure : la 2e fois, l'anti-doublons doit afficher « 0 nouvelles envoyées ».

In [70]:
!python src/collecte.py


Downloading:   0%|          |
Downloading: 100%|##########|
Downloading: 100%|##########|

Downloading:   0%|          |
Downloading: 100%|##########|
Downloading: 100%|##########|

Downloading:   0%|          |
Downloading: 100%|##########|

Downloading:   0%|          |
Downloading: 100%|##########|
Downloading: 100%|##########|



100%|##########| 1/1 [00:00<00:00, 2761.23it/s]

100%|##########| 1/1 [00:00<00:00, 4865.78it/s]

100%|##########| 1/1 [00:00<00:00, 1099.71it/s]

100%|##########| 1/1 [00:00<00:00, 4718.00it/s]

100%|##########| 1/1 [00:00<00:00, 1626.33it/s]

100%|##########| 1/1 [00:00<00:00, 1508.74it/s]

100%|##########| 1/1 [00:00<00:00, 1735.33it/s]

100%|##########| 1/1 [00:00<00:00, 2796.20it/s]


In [69]:
# Les 15 dernières lignes du log
lignes = open("logs/collecte.log", encoding="utf-8").read().splitlines()
print("\n".join(lignes[-15:]))

2026-10-17 20:54:55,651 | INFO | Appel API station_information.json : code HTTP 200
2026-10-17 20:54:55,806 | INFO | Appel API station_status.json : code HTTP 200
2026-10-17 20:55:00,341 | INFO | 
1519 out of 1519 rows loaded.
2026-10-17 20:55:00,342 | INFO | 1516 lignes reçues, 0 nouvelles envoyées
2026-10-17 20:55:02,031 | INFO | 
1 out of 1 rows loaded.
2026-10-17 20:55:02,031 | INFO | Début de la collecte
2026-10-17 20:55:02,174 | INFO | Appel API station_information.json : code HTTP 200
2026-10-17 20:55:02,345 | INFO | Appel API station_status.json : code HTTP 200
2026-10-17 20:55:05,087 | INFO | 
1519 out of 1519 rows loaded.
2026-10-17 20:55:05,087 | INFO | 1516 lignes reçues, 0 nouvelles envoyées
2026-10-17 20:55:08,350 | INFO | 
1 out of 1 rows loaded.


## 8. Automatisation

### 8.1 Première version : Planificateur de tâches Windows

In [71]:
%%writefile lancer_collecte.bat
@echo off
cd /d "C:\Users\utilisateur\velib suz"
call "C:\Users\utilisateur\anaconda3\Scripts\activate.bat"
python src\collecte.py

Overwriting lancer_collecte.bat


In [72]:
!schtasks /change /tn "Collecte Velib" /disable

Information : la tâche planifiée "Collecte Velib" a déjà été désactivée.
Opération réussie : les paramètres de la tâche planifiée "Collecte Velib"
                    ont été modifiés.


### 8.2 Version finale : GitHub Actions
Le workflow lance la collecte **toutes les heures à HH:05 (UTC)** sur les serveurs de GitHub, même PC éteint.

**Connexion sans clé :** la création de clés JSON étant bloquée par une règle de sécurité Google Cloud, la connexion passe par la **fédération d'identité** (Workload Identity Federation) : GitHub prouve son identité à Google, qui accorde un accès temporaire au compte de service `velib-collecte-15`. Une condition limite l'accès au seul dépôt `suzy5670/velocite-velib`. **Aucun secret n'est stocké sur GitHub.**

In [73]:
import os
os.makedirs(".github/workflows", exist_ok=True)

In [74]:
%%writefile .github/workflows/collecte.yml
name: Collecte Velib

on:
  schedule:
    - cron: "5 * * * *"   # toutes les heures, à HH:05 (heure UTC)
  workflow_dispatch:       # bouton pour lancer à la main

permissions:
  contents: read
  id-token: write          # autorise GitHub à prouver son identité à Google

jobs:
  collecte:
    runs-on: ubuntu-latest
    steps:
      - name: Récupérer le code
        uses: actions/checkout@v4

      - name: Installer Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Installer les bibliothèques
        run: pip install -r requirements.txt

      - name: Connexion à Google Cloud (sans clé)
        uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: projects/50689790778/locations/global/workloadIdentityPools/github/providers/github-actions
          service_account: velib-collecte-15@velocite-velib.iam.gserviceaccount.com

      - name: Lancer la collecte
        run: python src/collecte.py

      - name: Afficher les logs
        if: always()
        run: cat logs/collecte.log

Overwriting .github/workflows/collecte.yml


## 9. Requêtes SQL
Testées dans BigQuery, puis enregistrées dans le dossier `sql/`.

In [75]:
import os
os.makedirs("sql", exist_ok=True)

### Contrôle qualité : doublons
Si `nb_lignes` = `nb_cles_uniques`, chaque photo (station + heure) est unique.

In [76]:
%%writefile sql/00_controle_doublons.sql
-- Contrôle qualité : vérifie qu'il n'y a aucun doublon
-- Si nb_lignes = nb_cles_uniques, chaque photo (station + heure) est unique
SELECT
  COUNT(*) AS nb_lignes,
  COUNT(DISTINCT cle) AS nb_cles_uniques
FROM `velocite-velib.velib.disponibilites`

Overwriting sql/00_controle_doublons.sql


### Requête 1 – Stations vides maintenant
Filtre + jointure + tri : stations sans vélo lors de la dernière collecte, les plus grandes d'abord.

In [77]:
%%writefile sql/01_stations_vides_maintenant.sql
-- Requête 1 : stations vides lors de la dernière collecte, les plus grandes d'abord
SELECT s.name, s.stationCode, s.capacity, d.date_collecte
FROM `velocite-velib.velib.disponibilites` AS d
JOIN `velocite-velib.velib.stations` AS s ON d.station_id = s.station_id
WHERE d.station_vide
  AND d.date_collecte = (SELECT MAX(date_collecte) FROM `velocite-velib.velib.disponibilites`)
ORDER BY s.capacity DESC

Overwriting sql/01_stations_vides_maintenant.sql


### Requête 2 – Situation par heure
Agrégation : remplissage moyen et part de stations vides ou pleines pour chaque heure (heure de Paris). Répond à la question métier.

In [78]:
%%writefile sql/02_situation_par_heure.sql
-- Requête 2 : situation moyenne par heure (heure de Paris)
SELECT
  EXTRACT(HOUR FROM DATETIME(date_collecte, "Europe/Paris")) AS heure,
  ROUND(100 * AVG(taux_remplissage), 1) AS taux_remplissage_moyen_pct,
  ROUND(100 * COUNTIF(station_vide) / COUNT(*), 1) AS pct_stations_vides,
  ROUND(100 * COUNTIF(station_pleine) / COUNT(*), 1) AS pct_stations_pleines
FROM `velocite-velib.velib.disponibilites`
GROUP BY heure
ORDER BY heure

Overwriting sql/02_situation_par_heure.sql


### Requête 3 – Top 10 des stations le plus souvent vides
Jointure + agrégation + tri. Regroupement par `station_id` (et pas seulement par nom).

In [79]:
%%writefile sql/03_top10_stations_vides.sql
-- Requête 3 : top 10 des stations le plus souvent vides
SELECT
  s.station_id, s.name, s.stationCode,
  COUNT(*) AS nb_photos,
  COUNTIF(d.station_vide) AS nb_fois_vide,
  ROUND(100 * COUNTIF(d.station_vide) / COUNT(*), 1) AS pct_temps_vide
FROM `velocite-velib.velib.disponibilites` AS d
JOIN `velocite-velib.velib.stations` AS s ON d.station_id = s.station_id
GROUP BY s.station_id, s.name, s.stationCode
ORDER BY pct_temps_vide DESC, nb_fois_vide DESC
LIMIT 10

Overwriting sql/03_top10_stations_vides.sql


### Requête 4 – KPI du réseau
État du réseau lors de la dernière collecte : vélos disponibles (mécaniques / électriques) et places hors service.

In [80]:
%%writefile sql/04_kpi_reseau.sql
-- Requête 4 : état du réseau lors de la dernière collecte
SELECT
  SUM(d.num_bikes_available) AS velos_disponibles,
  SUM(d.nb_mecaniques) AS velos_mecaniques,
  SUM(d.nb_electriques) AS velos_electriques,
  SUM(GREATEST(s.capacity - d.num_bikes_available - d.num_docks_available, 0)) AS places_hors_service
FROM `velocite-velib.velib.disponibilites` AS d
JOIN `velocite-velib.velib.stations` AS s ON d.station_id = s.station_id
WHERE d.date_collecte = (SELECT MAX(date_collecte) FROM `velocite-velib.velib.disponibilites`)

Overwriting sql/04_kpi_reseau.sql


### Requête 5 – Secteurs les plus demandés
Départs estimés par zone : chaque photo est comparée à la précédente (`LAG`). Une baisse du nombre de vélos = des départs. Estimation minimale (flux nets par heure).

In [81]:
%%writefile sql/05_secteurs_plus_demandes.sql
-- Requête 5 : secteurs les plus demandés (départs nets estimés entre deux photos)
WITH variations AS (
  SELECT
    station_id,
    num_bikes_available
      - LAG(num_bikes_available) OVER (PARTITION BY station_id ORDER BY date_collecte) AS variation
  FROM `velocite-velib.velib.disponibilites`
),
zones AS (
  SELECT
    station_id,
    IF(SAFE_CAST(stationCode AS INT64) < 21000,
       CONCAT("Paris ", CAST(DIV(SAFE_CAST(stationCode AS INT64), 1000) AS STRING)),
       "Hors Paris") AS zone
  FROM `velocite-velib.velib.stations`
)
SELECT
  z.zone,
  SUM(IF(v.variation < 0, -v.variation, 0)) AS departs_estimes,
  SUM(IF(v.variation > 0, v.variation, 0)) AS arrivees_estimees
FROM variations AS v
JOIN zones AS z ON v.station_id = z.station_id
GROUP BY z.zone
ORDER BY departs_estimes DESC
LIMIT 10

Overwriting sql/05_secteurs_plus_demandes.sql


### Vue pour le dashboard
Joint les deux tables et prépare les champs du dashboard : heure de Paris, zone, indicateurs 0/1, places hors service, coordonnées GPS.

In [82]:
%%writefile sql/06_vue_dashboard.sql
-- Vue utilisée par le dashboard Data Studio
CREATE OR REPLACE VIEW `velocite-velib.velib.vue_dashboard` AS
SELECT
  d.*,
  s.name, s.stationCode, s.capacity, s.accepte_cb,
  DATETIME(d.date_collecte, "Europe/Paris") AS date_collecte_paris,
  EXTRACT(HOUR FROM DATETIME(d.date_collecte, "Europe/Paris")) AS heure,
  IF(SAFE_CAST(s.stationCode AS INT64) < 21000,
     CONCAT("Paris ", CAST(DIV(SAFE_CAST(s.stationCode AS INT64), 1000) AS STRING)),
     "Hors Paris") AS zone,
  GREATEST(s.capacity - d.num_bikes_available - d.num_docks_available, 0) AS places_hors_service,
  CAST(d.station_vide AS INT64) AS est_vide,
  CAST(d.station_pleine AS INT64) AS est_pleine,
  CONCAT(CAST(s.lat AS STRING), ",", CAST(s.lon AS STRING)) AS coordonnees
FROM `velocite-velib.velib.disponibilites` AS d
JOIN `velocite-velib.velib.stations` AS s ON d.station_id = s.station_id

Overwriting sql/06_vue_dashboard.sql


### Preuve 1 – Historique des exécutions

In [83]:
%%writefile sql/07_preuve_journal.sql
-- Preuve : historique des exécutions automatiques
SELECT
  DATETIME(date_execution, "Europe/Paris") AS execution_paris,
  statut, nb_lignes, message
FROM `velocite-velib.velib.journal_executions`
ORDER BY date_execution DESC

Overwriting sql/07_preuve_journal.sql


### Preuve 2 – Une photo par heure

In [84]:
%%writefile sql/08_preuve_photos_par_heure.sql
-- Preuve : nombre de stations enregistrées pour chaque heure de collecte
SELECT
  DATETIME_TRUNC(DATETIME(date_collecte, "Europe/Paris"), HOUR) AS heure_paris,
  COUNT(*) AS nb_stations
FROM `velocite-velib.velib.disponibilites`
GROUP BY heure_paris
ORDER BY heure_paris DESC

Overwriting sql/08_preuve_photos_par_heure.sql


## 10. Publication sur GitHub
Dépôt public : https://github.com/suzy5670/velocite-velib

### 10.1 Ce qui ne doit jamais être publié
Les secrets (`*.json`, `.env`), les logs et les fichiers temporaires de Jupyter.

In [85]:
%%writefile .gitignore
# Secrets : ne jamais publier
*.json
.env

# Fichiers générés automatiquement
logs/
__pycache__/
.ipynb_checkpoints/

Overwriting .gitignore


### 10.2 Bibliothèques du projet

In [86]:
%%writefile requirements.txt
requests==2.34.2
pandas==2.3.3
pandas-gbq==0.35.2

Overwriting requirements.txt


### 10.3 Mise en place du dépôt (déjà faite, pour mémoire)
```bash
git init
git branch -M main
git config user.name "suzy5670"
git remote add origin https://github.com/suzy5670/velocite-velib.git
git push -u origin main
```

### 10.4 Vérifier puis envoyer une nouvelle version
Toujours vérifier avec `git status` qu'aucun fichier sensible n'est listé avant d'envoyer.

In [87]:
!git status

On branch main
Your branch is ahead of 'origin/main' by 2 commits.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   src/collecte.py
	modified:   velib_propre.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


In [89]:
!git add .
!git commit -m "Notebook reorganise, requetes SQL corrigees"
!git push

On branch main
Your branch is ahead of 'origin/main' by 3 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


To https://github.com/suzy5670/velocite-velib.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/suzy5670/velocite-velib.git'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


## 11. Limites et pistes d'amélioration
- L'API donne l'état des stations, **pas les locations** : les départs par secteur sont des **estimations** (flux nets entre deux photos).
- Les vélos ne sont pas suivis individuellement : impossible de mesurer les vélos perdus ou non rendus.
- Quelques heures manquent au début (planificateur Windows), comblées depuis le passage à GitHub Actions.
- Bac à sable BigQuery : tables conservées 60 jours, pas de requêtes `INSERT` / `MERGE` (anti-doublons géré en Python).
- Pistes : alertes automatiques quand une station reste vide plusieurs heures, ajout de la météo, prévision de la demande.